In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import ...
import sys
import math
import polars as pl

# from ... import...
from pathlib import Path
from IPython.display import HTML, display

# get path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(10)
pl.Config.set_float_precision(6)

# from ... import ...
from src import (experiment_spec_from_profile,load_profile_bundle, run_convergence_study, run_frequency_sweep, run_parameter_sensitivity, run_whole_dragonfly_experiment)
from src.visualization import animate_whole_dragonfly_3d, plot_dragonfly_3d, plot_whole_dragonfly_3d

In [ ]:
bundle = load_profile_bundle(PROJECT_ROOT)
spec = experiment_spec_from_profile(bundle)

print('species:', bundle.profile.dragonfly.name)
print('wing:', bundle.wing.wing_type, bundle.wing.side)
print('output position, body [m]:', bundle.output_position_body_m)
print('source position, body [m]:', bundle.source_position_body_m)
print('drive [Hz, dB]:', spec.frequency_hz, bundle.profile.simulation.drive.spl_db_at_reference)
print('mode count:', spec.vibration.n_modes)

acoustics = bundle.profile.acoustics
module_geometry_frame = pl.DataFrame(
    {
        'opening_to_target_mm': [math.dist(bundle.source_position_m, bundle.output_position_m) * 1e3],
        'module_vertical_mm': [acoustics.pcb_size_vertical_mm],
        'module_horizontal_mm': [acoustics.pcb_size_horizontal_mm],
        'module_height_mm': [acoustics.pcb_size_height_mm],
        'buzzer_diameter_mm': [acoustics.pcb_buzzer_size_diameter_mm],
        'buzzer_height_mm': [acoustics.pcb_buzzer_size_height_mm],
        'height_without_buzzer_mm': [acoustics.pcb_size_without_buzzer_height_mm],
    }
)
display(module_geometry_frame)
display(plot_dragonfly_3d(bundle))

In [ ]:
whole_result = run_whole_dragonfly_experiment(bundle, spec)
maximum_wing = whole_result.maximum_wing
result = maximum_wing.simulation
peak_index = max(range(len(result.time_s)), key=lambda index: abs(result.output_displacement_m[index]))
nearest_mode = min(result.modes, key=lambda mode: abs(mode.natural_frequency_hz - result.frequency_hz))

print('integration:', result.integration_method)
print('samples / actual dt:', len(result.time_s), result.time_step_s)
print('largest-response wing:', maximum_wing.wing.wing_type, maximum_wing.wing.side)
print('wing SPL range [dB]:', result.pressure_field.spl_range_db)
print('nearest mode [index, Hz]:', nearest_mode.mode_index, nearest_mode.natural_frequency_hz)
print('maximum physical output displacement [m]:', maximum_wing.peak_output_displacement_m)
wing_result_frame = pl.DataFrame({
    'wing': [f'{item.wing.wing_type}_{item.wing.side}' for item in whole_result.wings],
    'illuminated': [item.is_illuminated for item in whole_result.wings],
    'first_natural_frequency_hz': [item.simulation.modes[0].natural_frequency_hz for item in whole_result.wings],
    'minimum_spl_db': [item.simulation.pressure_field.spl_range_db[0] for item in whole_result.wings],
    'maximum_spl_db': [item.simulation.pressure_field.spl_range_db[1] for item in whole_result.wings],
    'peak_output_um': [item.peak_output_displacement_m * 1e6 for item in whole_result.wings],
})
display(wing_result_frame)
display(plot_whole_dragonfly_3d(bundle, whole_result, sample_index=peak_index, deformation_scale=100.0))

In [ ]:
frame_step = max(1, len(whole_result.time_s) // 80)
dragonfly_animation = animate_whole_dragonfly_3d(
    bundle, whole_result, deformation_scale=100.0, frame_step=frame_step
)
display(HTML(dragonfly_animation.to_jshtml()))

In [ ]:
convergence = run_convergence_study(bundle, spec)
convergence_frame = pl.DataFrame(
    {
        'parameter': [sample.parameter for sample in convergence.samples],
        'value': [sample.value for sample in convergence.samples],
        'peak_output_um': [sample.peak_output_displacement_m * 1e6 for sample in convergence.samples],
        'steady_output_rms_um': [sample.steady_output_displacement_rms_m * 1e6 for sample in convergence.samples],
        'relative_peak_error_percent': [sample.relative_peak_error * 100 for sample in convergence.samples],
        'reference': [sample.is_reference for sample in convergence.samples],
    }
)
display(convergence_frame)

In [ ]:
frequencies_hz = tuple(sorted(
    {float(value) for value in range(50, 3501, 25)}
    | {mode.natural_frequency_hz for mode in whole_result.selected_wing.simulation.modes if mode.natural_frequency_hz <= 3500.0}
))
frequency_sweep = run_frequency_sweep(bundle, spec, frequencies_hz)
frequency_response_frame = pl.DataFrame(
    {
        'frequency_hz': [sample.frequency_hz for sample in frequency_sweep.samples],
        'tip_rms_um': [sample.tip_displacement_rms_m * 1e6 for sample in frequency_sweep.samples],
        'output_rms_um': [sample.output_displacement_rms_m * 1e6 for sample in frequency_sweep.samples],
        'output_phase_rad': [sample.output_phase_rad for sample in frequency_sweep.samples],
        'small_deflection': [sample.output_displacement_rms_m < 0.05 * bundle.wing.length_m for sample in frequency_sweep.samples],
    }
)
largest_frequency_responses = frequency_response_frame.sort('output_rms_um', descending=True).head(5)
display(largest_frequency_responses)

sensitivity = run_parameter_sensitivity(bundle, spec)
sensitivity_frame = (
    pl.DataFrame(
        {
            'parameter': [sample.parameter for sample in sensitivity.samples],
            'factor': [sample.factor for sample in sensitivity.samples],
            'value': [sample.value for sample in sensitivity.samples],
            'output_rms_um': [sample.output_displacement_rms_m * 1e6 for sample in sensitivity.samples],
            'ratio_to_nominal': [sample.ratio_to_nominal for sample in sensitivity.samples],
        }
    )
    .with_columns(((pl.col('ratio_to_nominal') - 1.0) * 100.0).alias('change_percent'))
)
display(sensitivity_frame)